In [ ]:
import torch
import pandas as pd

from src.image_model import ImageOnlyFakeNewsDetectionModel
from src.image_dataloader import create_image_dataloader

In [ ]:
print("PyTorch Version :", torch.__version__)
print("CUDA Available  :", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using Device :", device)

if device.type == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))

In [ ]:
BATCH_SIZE = 32
NUM_EPOCHS = 10
LEARNING_RATE = 1e-4

In [ ]:
train_loader = create_image_dataloader(
    csv_path="./data/processed/train_clean.csv",
    images_dir="./images",
    batch_size=BATCH_SIZE,
    shuffle=True,
    train=True,
)

val_loader = create_image_dataloader(
    csv_path="./data/processed/validation_clean.csv",
    images_dir="./images",
    batch_size=BATCH_SIZE,
    shuffle=False,
    train=False,
)

test_loader = create_image_dataloader(
    csv_path="./data/processed/test_clean.csv",
    images_dir="./images",
    batch_size=BATCH_SIZE,
    shuffle=False,
    train=False,
)

In [ ]:
sample = next(iter(train_loader))

print(sample["image"].shape)
print(sample["label"].shape)

In [ ]:
model = ImageOnlyFakeNewsDetectionModel()
model = model.to(device)

print(next(model.parameters()).device)

In [ ]:
batch = next(iter(train_loader))

image = batch["image"].to(device)
label = batch["label"].to(device)

model.eval()

with torch.no_grad():
    logits = model(image)
    loss = model.loss_fn(logits, label)

print("Logits shape:", logits.shape)
print("Loss:", loss.item())

In [ ]:
import pytorch_lightning as pl

from pytorch_lightning.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
)

In [ ]:
checkpoint_callback = ModelCheckpoint(
    monitor="val_acc",
    mode="max",
    save_top_k=1,
    filename="best-image-model",
)

In [ ]:
early_stop_callback = EarlyStopping(
    monitor="val_loss",
    patience=3,
    mode="min",
)

In [ ]:
trainer = pl.Trainer(
    max_epochs=NUM_EPOCHS,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    callbacks=[
        checkpoint_callback,
        early_stop_callback,
    ],
    log_every_n_steps=10,
)

In [ ]:
model = ImageOnlyFakeNewsDetectionModel()

trainer.fit(
    model,
    train_dataloaders=train_loader,
    val_dataloaders=val_loader,
)

In [ ]:
best_model = ImageOnlyFakeNewsDetectionModel.load_from_checkpoint(
    checkpoint_callback.best_model_path
)

In [ ]:
trainer.test(
    model=best_model,
    dataloaders=test_loader,
)

## Metrics (accuracy, precision, recall, F1, confusion matrix, ROC-AUC)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
)

best_model.eval()

y_true = []
y_pred = []
y_prob = []

device = next(best_model.parameters()).device

with torch.no_grad():
    for batch in test_loader:

        image = batch["image"].to(device)
        labels = batch["label"].to(device)

        logits = best_model(image)

        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(logits, dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())
        y_prob.extend(probs[:, 1].cpu().numpy())  # probability of class 1

In [ ]:
print("Accuracy  :", accuracy_score(y_true, y_pred))
print("Precision :", precision_score(y_true, y_pred))
print("Recall    :", recall_score(y_true, y_pred))
print("F1-score  :", f1_score(y_true, y_pred))
print("ROC-AUC   :", roc_auc_score(y_true, y_prob))
print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

## Export image embeddings (for the fusion model)

In [ ]:
import os


def create_image_embeddings(df, loader):

    embedding_dict = {}

    best_model.eval()

    with torch.no_grad():
        for batch in loader:

            ids = batch["id"]
            image = batch["image"].to(device)

            features = best_model.extract_features(image)

            for image_id, embedding in zip(ids, features):
                embedding_dict[image_id] = embedding.cpu()

    return embedding_dict


train_df = pd.read_csv("./data/processed/train_clean.csv")
val_df = pd.read_csv("./data/processed/validation_clean.csv")
test_df = pd.read_csv("./data/processed/test_clean.csv")

eval_train_loader = create_image_dataloader(
    csv_path="./data/processed/train_clean.csv",
    images_dir="./images",
    batch_size=BATCH_SIZE,
    shuffle=False,
    train=False,
)

train_image_embeddings = create_image_embeddings(train_df, eval_train_loader)
val_image_embeddings = create_image_embeddings(val_df, val_loader)
test_image_embeddings = create_image_embeddings(test_df, test_loader)

os.makedirs("./data/embeddings", exist_ok=True)

torch.save(train_image_embeddings, "./data/embeddings/train_image_embeddings.pt")
torch.save(val_image_embeddings, "./data/embeddings/validation_image_embeddings.pt")
torch.save(test_image_embeddings, "./data/embeddings/test_image_embeddings.pt")

print("Saved image embeddings.")